# Tarea 3: Metodologia de Modelamiento

Este notebook implementa la fase completa de modelamiento para el laboratorio de prediccion y clasificacion en la industria azucarera. Se reutilizan las decisiones de limpieza y seleccion de variables tomadas en la Tarea 2, pero el flujo se reconstruye de forma minima para que el notebook sea independiente y ejecutable de arriba hacia abajo.

Problemas a resolver:
- Regresion de `TCH`
- Regresion de `%Sac.Caña`
- Clasificacion de `TCH` en `bajo`, `medio` y `alto`
- Clasificacion de `%Sac.Caña` en `bajo`, `medio` y `alto`

Nota metodologica: en esta version rapida se conserva Random Forest como modelo avanzado y se eliminan SVM/SVC y busquedas pesadas de hiperparametros para que el notebook sea ejecutable en tiempos razonables.

Criterios metodologicos clave:
- Se trabaja solo con variables justificadas en la Tarea 2.
- El encoding, la imputacion y el escalado se encapsulan en `Pipeline` para evitar leakage.
- Las clases para clasificacion se definen con terciles calculados solo sobre `y_train`.
- No se usan variables derivadas o de fuga como `TAH`, `Ton.Azucar`, `%ATR`, `KATRHM` o `Rdto`.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    f1_score,
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Found unknown categories.*", category=UserWarning)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


## 1. Carga y preparacion minima de datos

Se reconstruye solo la parte minima necesaria para modelar. En lugar de rehacer el EDA completo, se retoman las mismas decisiones de Tarea 2:
- Variables base: `Edad Ult Cos`, `Variedad`, `Suelo`, `Dosis Madurante`, `TonUltCorte`
- Variables descartadas por calidad/redundancia: `%Infest.Diatrea`, `Num.Riegos`, `Semanas mad.`, `Area Neta`
- `%Sac.Caña` elimina observaciones sin target observado

La imputacion se deja dentro de los `Pipeline` para mantener la regla de Tarea 2 sin contaminar el conjunto de prueba.


In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
OUTPUT_TABLES_DIR = BASE_DIR / "outputs" / "tables"
OUTPUT_FIGURES_DIR = BASE_DIR / "outputs" / "figures"
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "Edad Ult Cos",
    "Variedad",
    "Suelo",
    "Dosis Madurante",
    "TonUltCorte",
]

NUMERIC_FEATURES = ["Edad Ult Cos", "Dosis Madurante", "TonUltCorte"]
CATEGORICAL_FEATURES = ["Variedad", "Suelo"]
TARGETS = ["TCH", "%Sac.Caña"]

EXCLUDED_BY_EDA = ["%Infest.Diatrea", "Num.Riegos", "Semanas mad.", "Area Neta"]
LEAKAGE_FEATURES = ["TAH", "Ton.Azucar", "%ATR", "KATRHM", "Rdto"]

historico = pd.read_excel(RAW_DIR / "HISTORICO_SUERTES.xlsx")

def prepare_modeling_frame(data, target):
    frame = data.loc[:, FEATURES + [target]].copy()
    frame = frame.dropna(subset=[target]).reset_index(drop=True)
    return frame

df_tch = prepare_modeling_frame(historico, "TCH")
df_sac = prepare_modeling_frame(historico, "%Sac.Caña")

dataset_summary = pd.DataFrame(
    {
        "dataset": ["df_tch", "df_sac"],
        "target": ["TCH", "%Sac.Caña"],
        "filas": [len(df_tch), len(df_sac)],
        "columnas": [df_tch.shape[1], df_sac.shape[1]],
    }
)

display(dataset_summary)
display(df_tch.head())
print("Variables excluidas por EDA:", EXCLUDED_BY_EDA)
print("Variables descartadas por fuga de informacion:", LEAKAGE_FEATURES)


## 2. Preprocesamiento y funciones auxiliares

Se crean dos preprocessors:
- `preprocessor_base`: imputacion + one-hot encoding
- `preprocessor_scaled`: imputacion + one-hot encoding + escalado de variables numericas

Esto permite usar escalado solo en algoritmos sensibles a la escala, como KNN y regresion logistica.


In [ ]:
REGRESSION_SCORING = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
}

CLASSIFICATION_SCORING = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "kappa": make_scorer(cohen_kappa_score),
}

REGRESSION_CV = KFold(n_splits=3, shuffle=True, random_state=42)
CLASSIFICATION_CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", drop="first", sparse=False)

def make_preprocessor(scale_numeric=False):
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(steps=numeric_steps)

    return ColumnTransformer(
        transformers=[
            ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
            ("num", numeric_pipeline, NUMERIC_FEATURES),
        ]
    )

def make_pipeline(model, scale_numeric=False):
    return Pipeline(
        steps=[
            ("preprocessor", make_preprocessor(scale_numeric=scale_numeric)),
            ("model", model),
        ]
    )

def strip_transformer_prefix(feature_name):
    return feature_name.split("__", 1)[1] if "__" in feature_name else feature_name

def get_feature_names_from_pipeline(fitted_pipeline):
    preprocessor = fitted_pipeline.named_steps["preprocessor"]
    try:
        raw_names = preprocessor.get_feature_names_out()
    except AttributeError:
        raw_names = []
        for _, transformer, columns in preprocessor.transformers_:
            if hasattr(transformer, "get_feature_names_out"):
                try:
                    names = transformer.get_feature_names_out(columns)
                except TypeError:
                    names = transformer.get_feature_names_out()
            elif hasattr(transformer, "named_steps"):
                last_step = list(transformer.named_steps.values())[-1]
                if hasattr(last_step, "get_feature_names_out"):
                    try:
                        names = last_step.get_feature_names_out(columns)
                    except TypeError:
                        names = last_step.get_feature_names_out()
                else:
                    names = columns
            else:
                names = columns
            raw_names.extend(names)

    return np.array([strip_transformer_prefix(name) for name in raw_names], dtype=object)

def summarize_cv_scores(cv_scores, regression=True):
    summary = {}
    for metric_name, values in cv_scores.items():
        if not metric_name.startswith("test_"):
            continue
        clean_name = metric_name.replace("test_", "")
        metric_value = float(np.mean(values))
        if regression and clean_name in {"rmse", "mae"}:
            metric_value *= -1
        summary[f"cv_{clean_name}"] = metric_value
    return summary

def build_tercile_targets(y_train_cont, y_test_cont):
    lower_cut, upper_cut = np.quantile(y_train_cont, [1 / 3, 2 / 3])
    if np.isclose(lower_cut, upper_cut):
        raise ValueError("Los terciles coinciden y no permiten crear tres clases estables.")

    bins = [-np.inf, lower_cut, upper_cut, np.inf]
    labels = ["bajo", "medio", "alto"]

    y_train_cls = pd.Series(
        pd.cut(y_train_cont, bins=bins, labels=labels, include_lowest=True).astype(str),
        index=y_train_cont.index,
    )
    y_test_cls = pd.Series(
        pd.cut(y_test_cont, bins=bins, labels=labels, include_lowest=True).astype(str),
        index=y_test_cont.index,
    )

    return y_train_cls, y_test_cls, {"q33": lower_cut, "q66": upper_cut}

def evaluate_regression_model(model_name, estimator, X_train, y_train, X_test, y_test, search=None):
    if search is not None:
        search.fit(X_train, y_train)
        fitted_model = search.best_estimator_
        best_params = search.best_params_
    else:
        fitted_model = estimator.fit(X_train, y_train)
        best_params = {}

    cv_scores = cross_validate(
        fitted_model,
        X_train,
        y_train,
        cv=REGRESSION_CV,
        scoring=REGRESSION_SCORING,
        n_jobs=1,
    )

    y_pred = fitted_model.predict(X_test)
    result = {
        "modelo": model_name,
        **summarize_cv_scores(cv_scores, regression=True),
        "test_r2": r2_score(y_test, y_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "test_mae": mean_absolute_error(y_test, y_pred),
    }
    return result, fitted_model, y_pred, best_params

def evaluate_classification_model(model_name, estimator, X_train, y_train, X_test, y_test, search=None):
    if search is not None:
        search.fit(X_train, y_train)
        fitted_model = search.best_estimator_
        best_params = search.best_params_
    else:
        fitted_model = estimator.fit(X_train, y_train)
        best_params = {}

    cv_scores = cross_validate(
        fitted_model,
        X_train,
        y_train,
        cv=CLASSIFICATION_CV,
        scoring=CLASSIFICATION_SCORING,
        n_jobs=1,
    )

    y_pred = fitted_model.predict(X_test)
    result = {
        "modelo": model_name,
        **summarize_cv_scores(cv_scores, regression=False),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "test_recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "test_f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "test_kappa": cohen_kappa_score(y_test, y_pred),
    }
    report_text = classification_report(y_test, y_pred, zero_division=0)
    return result, fitted_model, y_pred, report_text, best_params

def make_safe_filename(title):
    replacements = {
        "%": "pct",
        "ñ": "n",
        "Ñ": "n",
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
    }
    clean_title = title
    for old, new in replacements.items():
        clean_title = clean_title.replace(old, new)
    clean_title = clean_title.lower()
    clean_title = "".join(char if char.isalnum() else "_" for char in clean_title)
    clean_title = "_".join(part for part in clean_title.split("_") if part)
    return f"{clean_title}.png"

def save_figure(title):
    output_path = OUTPUT_FIGURES_DIR / make_safe_filename(title)
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    print(f"Figura guardada: {output_path}")
    return output_path

def plot_regression_predictions(y_true, y_pred, title):
    plt.figure(figsize=(7, 6))
    plt.scatter(y_true, y_pred, alpha=0.35)
    lower = min(np.min(y_true), np.min(y_pred))
    upper = max(np.max(y_true), np.max(y_pred))
    plt.plot([lower, upper], [lower, upper], color="crimson", linestyle="--", linewidth=2)
    plt.xlabel("Valor real")
    plt.ylabel("Prediccion")
    plt.title(title)
    plt.tight_layout()
    save_figure(title)
    plt.show()

def plot_confusion_matrix(y_true, y_pred, title):
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        labels=["bajo", "medio", "alto"],
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(title)
    plt.tight_layout()
    save_figure(title)
    plt.show()

def group_feature_importance(fitted_pipeline):
    feature_names = get_feature_names_from_pipeline(fitted_pipeline)
    importances = fitted_pipeline.named_steps["model"].feature_importances_
    importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})

    def collapse_feature_name(name):
        if name.startswith("Variedad_"):
            return "Variedad"
        if name.startswith("Suelo_"):
            return "Suelo"
        return name

    grouped = (
        importance_df.assign(variable=importance_df["feature"].map(collapse_feature_name))
        .groupby("variable", as_index=False)["importance"]
        .sum()
        .sort_values("importance", ascending=False)
    )
    return grouped

def plot_grouped_importance(importance_df, title):
    plt.figure(figsize=(8, 5))
    sns.barplot(data=importance_df, x="importance", y="variable", palette="viridis")
    plt.title(title)
    plt.xlabel("Importancia acumulada")
    plt.ylabel("Variable")
    plt.tight_layout()
    save_figure(title)
    plt.show()


## 3. Modelos de regresion

Se usa un benchmark lineal y un modelo avanzado rapido:
- `LinearRegression` como linea base interpretable
- `RandomForestRegressor` para capturar no linealidades e interacciones

Se omite `SVR` en esta version porque su costo con kernel RBF es alto para el tamano del dataset.


In [ ]:
regression_datasets = {"TCH": df_tch, "%Sac.Caña": df_sac}
regression_results = []
regression_artifacts = {}

for target, frame in regression_datasets.items():
    X = frame[FEATURES]
    y = frame[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
    )

    regression_artifacts[target] = {
        "split": {
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test,
        }
    }

    regression_configs = {
        "LinearRegression": {
            "pipeline": make_pipeline(LinearRegression(), scale_numeric=False),
            "search": None,
        },
        "RandomForestRegressor": {
            "pipeline": make_pipeline(
                RandomForestRegressor(
                    n_estimators=120,
                    max_depth=12,
                    min_samples_leaf=2,
                    random_state=42,
                    n_jobs=-1,
                ),
                scale_numeric=False,
            ),
            "search": None,
        },
    }

    for model_name, config in regression_configs.items():
        pipeline = config["pipeline"]
        search = config["search"](pipeline) if config["search"] else None
        X_model_train, y_model_train = X_train, y_train

        result, fitted_model, y_pred, best_params = evaluate_regression_model(
            model_name=model_name,
            estimator=pipeline,
            X_train=X_model_train,
            y_train=y_model_train,
            X_test=X_test,
            y_test=y_test,
            search=search,
        )

        result["target"] = target
        regression_results.append(result)
        regression_artifacts[target][model_name] = {
            "model": fitted_model,
            "y_test": y_test,
            "y_pred": y_pred,
            "best_params": best_params,
        }

resumen_regresion = pd.DataFrame(regression_results)[
    ["target", "modelo", "cv_r2", "cv_rmse", "cv_mae", "test_r2", "test_rmse", "test_mae"]
].sort_values(["target", "test_r2"], ascending=[True, False])

display(resumen_regresion.round(4))


In [ ]:
# En el modelo lineal, las dummies categorizadas se interpretan respecto a una categoria de referencia.
for target in regression_datasets:
    linear_pipeline = regression_artifacts[target]["LinearRegression"]["model"]
    feature_names = get_feature_names_from_pipeline(linear_pipeline)
    coef_df = pd.DataFrame(
        {
            "feature": feature_names,
            "coeficiente": linear_pipeline.named_steps["model"].coef_,
        }
    )
    coef_df["abs_coeficiente"] = coef_df["coeficiente"].abs()
    coef_df = coef_df.sort_values("abs_coeficiente", ascending=False)

    print(f"\nCoeficientes mas influyentes para {target}")
    display(coef_df[["feature", "coeficiente"]].head(15))


## 4. Modelos de clasificacion

Las clases `bajo`, `medio` y `alto` se construyen con terciles sobre `y_train` para no filtrar informacion del test. Se comparan:
- `LogisticRegression` con regularizacion L2 como linea base interpretable
- `KNeighborsClassifier` con parametros fijos como comparacion no parametrica rapida
- `RandomForestClassifier` como modelo no lineal robusto

Se omite `SVC` en esta version porque el entrenamiento con kernel RBF y validacion cruzada tarda demasiado para el tamano del dataset.


In [ ]:
classification_datasets = {"TCH": df_tch, "%Sac.Caña": df_sac}
classification_results = []
classification_artifacts = {}

for target, frame in classification_datasets.items():
    X = frame[FEATURES]
    y_cont = frame[target]

    X_train, X_test, y_train_cont, y_test_cont = train_test_split(
        X,
        y_cont,
        test_size=0.20,
        random_state=42,
    )

    y_train_cls, y_test_cls, cutpoints = build_tercile_targets(y_train_cont, y_test_cont)

    print(f"\n{target} -> cortes de terciles: q33={cutpoints['q33']:.4f}, q66={cutpoints['q66']:.4f}")
    class_distribution = pd.DataFrame(
        {
            "train": y_train_cls.value_counts().reindex(["bajo", "medio", "alto"]),
            "test": y_test_cls.value_counts().reindex(["bajo", "medio", "alto"]),
        }
    )
    display(class_distribution)

    classification_artifacts[target] = {
        "split": {
            "X_train": X_train,
            "X_test": X_test,
            "y_train_cont": y_train_cont,
            "y_test_cont": y_test_cont,
            "y_train_cls": y_train_cls,
            "y_test_cls": y_test_cls,
        },
        "cutpoints": cutpoints,
    }

    classification_configs = {
        "LogisticRegression": {
            "pipeline": make_pipeline(
                LogisticRegression(
                    solver="lbfgs",
                    penalty="l2",
                    max_iter=1000,
                    random_state=42,
                ),
                scale_numeric=True,
            ),
            "search": None,
        },
        "KNeighborsClassifier": {
            "pipeline": make_pipeline(
                KNeighborsClassifier(n_neighbors=11, weights="distance", p=2),
                scale_numeric=True,
            ),
            "search": None,
        },
        "RandomForestClassifier": {
            "pipeline": make_pipeline(
                RandomForestClassifier(
                    n_estimators=120,
                    max_depth=12,
                    min_samples_leaf=2,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
                scale_numeric=False,
            ),
            "search": None,
        },
    }

    for model_name, config in classification_configs.items():
        pipeline = config["pipeline"]
        search = config["search"](pipeline) if config["search"] else None
        X_model_train, y_model_train = X_train, y_train_cls

        result, fitted_model, y_pred, report_text, best_params = evaluate_classification_model(
            model_name=model_name,
            estimator=pipeline,
            X_train=X_model_train,
            y_train=y_model_train,
            X_test=X_test,
            y_test=y_test_cls,
            search=search,
        )

        result["target"] = target
        classification_results.append(result)
        classification_artifacts[target][model_name] = {
            "model": fitted_model,
            "y_test": y_test_cls,
            "y_pred": y_pred,
            "report_text": report_text,
            "best_params": best_params,
        }

resumen_clasificacion = pd.DataFrame(classification_results)[
    [
        "target",
        "modelo",
        "cv_accuracy",
        "cv_precision_macro",
        "cv_recall_macro",
        "cv_f1_macro",
        "cv_kappa",
        "test_accuracy",
        "test_precision_macro",
        "test_recall_macro",
        "test_f1_macro",
        "test_kappa",
    ]
].sort_values(["target", "test_f1_macro"], ascending=[True, False])

display(resumen_clasificacion.round(4))


In [ ]:
for target, artifacts in classification_artifacts.items():
    for model_name, details in artifacts.items():
        if model_name in {"split", "cutpoints"}:
            continue

        print(f"\nReporte de clasificacion - {target} - {model_name}")
        print(details["report_text"])
        plot_confusion_matrix(
            y_true=details["y_test"],
            y_pred=details["y_pred"],
            title=f"Matriz de confusion - {target} - {model_name}",
        )


## 5. Comparacion de modelos y validacion final

Se consolidan las metricas de todos los modelos para seleccionar el mejor por problema:
- Regresion: mayor `R2` en test y `RMSE` como desempate
- Clasificacion: mayor `F1_macro` en test

Tambien se comparan metricas de CV y test para vigilar sobreajuste.


In [ ]:
best_regression = (
    resumen_regresion.sort_values(["target", "test_r2", "test_rmse"], ascending=[True, False, True])
    .groupby("target", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_classification = (
    resumen_clasificacion.sort_values(["target", "test_f1_macro", "test_accuracy"], ascending=[True, False, False])
    .groupby("target", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_regression = best_regression.assign(
    gap_r2=lambda df: df["test_r2"] - df["cv_r2"],
    gap_rmse=lambda df: df["test_rmse"] - df["cv_rmse"],
)

best_classification = best_classification.assign(
    gap_f1_macro=lambda df: df["test_f1_macro"] - df["cv_f1_macro"],
    gap_accuracy=lambda df: df["test_accuracy"] - df["cv_accuracy"],
    gap_kappa=lambda df: df["test_kappa"] - df["cv_kappa"],
)

print("Resumen comparativo de regresion")
display(resumen_regresion.round(4))
print("Mejores modelos de regresion")
display(best_regression.round(4))

print("Resumen comparativo de clasificacion")
display(resumen_clasificacion.round(4))
print("Mejores modelos de clasificacion")
display(best_classification.round(4))

table_paths = {
    "resumen_regresion": OUTPUT_TABLES_DIR / "resumen_regresion.csv",
    "resumen_clasificacion": OUTPUT_TABLES_DIR / "resumen_clasificacion.csv",
    "mejores_regresion": OUTPUT_TABLES_DIR / "mejores_regresion.csv",
    "mejores_clasificacion": OUTPUT_TABLES_DIR / "mejores_clasificacion.csv",
}

resumen_regresion.to_csv(table_paths["resumen_regresion"], index=False)
resumen_clasificacion.to_csv(table_paths["resumen_clasificacion"], index=False)
best_regression.to_csv(table_paths["mejores_regresion"], index=False)
best_classification.to_csv(table_paths["mejores_clasificacion"], index=False)

print("Tablas guardadas/sobrescritas:")
for path in table_paths.values():
    print(path)


In [ ]:
for _, row in best_regression.iterrows():
    artifact = regression_artifacts[row["target"]][row["modelo"]]
    plot_regression_predictions(
        y_true=artifact["y_test"],
        y_pred=artifact["y_pred"],
        title=f"Prediccion vs real - {row['target']} - {row['modelo']}",
    )

for _, row in best_classification.iterrows():
    artifact = classification_artifacts[row["target"]][row["modelo"]]
    plot_confusion_matrix(
        y_true=artifact["y_test"],
        y_pred=artifact["y_pred"],
        title=f"Validacion final - {row['target']} - {row['modelo']}",
    )


## 6. Interpretabilidad con Random Forest

Se usa `feature_importances_` como referencia interpretable para modelos de arboles. Las importancias de columnas one-hot se agrupan nuevamente por variable original para que la lectura sea mas cercana al negocio.

Lectura esperada:
- Variables historicas como `TonUltCorte` suelen capturar inercia productiva.
- Variables agronomicas como `Edad Ult Cos` o `Dosis Madurante` suelen relacionarse con madurez y rendimiento.
- `Variedad` y `Suelo` pueden reflejar diferencias biologicas y edaficas relevantes.


In [ ]:
for target in regression_datasets:
    rf_regressor = regression_artifacts[target]["RandomForestRegressor"]["model"]
    rf_importance = group_feature_importance(rf_regressor)
    print(f"\nImportancia agrupada - Regresion - {target}")
    display(rf_importance.round(4))
    plot_grouped_importance(rf_importance, f"Importancia de variables - RF Regresion ({target})")

for target in classification_datasets:
    rf_classifier = classification_artifacts[target]["RandomForestClassifier"]["model"]
    rf_importance = group_feature_importance(rf_classifier)
    print(f"\nImportancia agrupada - Clasificacion - {target}")
    display(rf_importance.round(4))
    plot_grouped_importance(rf_importance, f"Importancia de variables - RF Clasificacion ({target})")

print("\nFiguras guardadas en:", OUTPUT_FIGURES_DIR)
for path in sorted(OUTPUT_FIGURES_DIR.glob("*.png")):
    print(path.name)
